In [1]:
import sys
from types import ModuleType

# 제거된 vertexai 모듈을 메모리에 가짜로 등록하여 라이브러리 충돌을 방지합니다.
dummy_chat_vertex = ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertex.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertex

dummy_llm_vertex = ModuleType("langchain_community.llms.vertexai")
dummy_llm_vertex.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = dummy_llm_vertex

In [2]:
import pandas as pd
from ragas import EvaluationDataset

eval_raw_data = {
    "user_input": [
        "도서관 주말 운영 시간과 대출 가능 권수는 어떻게 되나요?",
        "시험 기간 평일 도서관 마감 시간은 몇 시인가요?"
    ],
    "retrieved_contexts": [
        [
            "중앙도서관 운영 안내: 주말(토, 일)은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",
            "중앙도서관 매점 운영 안내: 평일 오전 9시부터 오후 6시까지 운영합니다."
        ],
        [
            "중앙도서관 특별 규정: 시험 기간 평일은 09시부터 22시까지 연장 운영합니다.",
            "도서관 분실물 보관소는 1층 로비 안내실에서 상시운영됩니다."
        ]
    ],
    "response": [
        # 문서에 명시된 사실만을 정확히 서술한 답변입니다. (높은 사실성 기대)
        "주말은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",
        # 문서에 없는 24시간 개방 내용을 추가하여 환각이 포함된 답변입니다. (낮은 사실성 유도)
        "시험 기간 평일 마감 시간은 22시까지입니다. 또한 주말에는 24시간 특별 개방을 실시합니다."
    ]
}

df_eval = pd.DataFrame(eval_raw_data)
ragas_dataset = EvaluationDataset.from_pandas(df_eval)

ragas_dataset

c:\Users\yonggyo\Documents\ax_study\12_rag\chapter06_07\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response'], len=2)

In [3]:
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="none",
    temperature=0
)

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

evaluator_llm = LangchainLLMWrapper(model)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

metrics = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
]

evaluation_result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    raise_exceptions=True
)


C:\Users\yonggyo\AppData\Local\Temp\ipykernel_8968\3738702579.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy
C:\Users\yonggyo\AppData\Local\Temp\ipykernel_8968\3738702579.py:4: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy
C:\Users\yonggyo\AppData\Local\Temp\ipykernel_8968\3738702579.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key

In [4]:
evaluation_result

{'faithfulness': 0.7500, 'answer_relevancy': 0.6228}

In [5]:
df_result = evaluation_result.to_pandas()

df_result

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,도서관 주말 운영 시간과 대출 가능 권수는 어떻게 되나요?,"[중앙도서관 운영 안내: 주말(토, 일)은 오전 10시부터 오후 5시까지 운영하며,...","주말은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",1.0,0.592771
1,시험 기간 평일 도서관 마감 시간은 몇 시인가요?,"[중앙도서관 특별 규정: 시험 기간 평일은 09시부터 22시까지 연장 운영합니다.,...",시험 기간 평일 마감 시간은 22시까지입니다. 또한 주말에는 24시간 특별 개방을 ...,0.5,0.652903


In [6]:
SCORE_THRESHOLD = 0.6
failed_cases = df_result[(df_result['faithfulness'] < SCORE_THRESHOLD) | (df_result['answer_relevancy'] < SCORE_THRESHOLD)].copy()

failed_cases

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,도서관 주말 운영 시간과 대출 가능 권수는 어떻게 되나요?,"[중앙도서관 운영 안내: 주말(토, 일)은 오전 10시부터 오후 5시까지 운영하며,...","주말은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",1.0,0.592771
1,시험 기간 평일 도서관 마감 시간은 몇 시인가요?,"[중앙도서관 특별 규정: 시험 기간 평일은 09시부터 22시까지 연장 운영합니다.,...",시험 기간 평일 마감 시간은 22시까지입니다. 또한 주말에는 24시간 특별 개방을 ...,0.5,0.652903


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    """
    당신은 RAG 생성 품질을 전문적으로 진단하는 전문가 입니다.
    아래의 질문, 참고 문서, 생성된 답변을 대조하여 오류를 분석하세요.

    [질문]: {question}
    [생성 답변]: {response}
    [참고 문서]
    {contexts}

    답안이 참고 문서와 온전히 부합하면 '오류 없음'으로 판정하고, 결함이 있으면 아래 항목 맞춰 간결하게 작성하세요.
    1. 판정 결과: [오류 없음 / 환각 / 문맥 무시 / 프롬프트 지시 불이행 / 기타 중 택1]
    2. 상세 원인: [실패원인을 2줄 이내로 명확하게 진술]
    3. 품질 개선안: [해결을 위한 구체적인 조치사항을 2줄 이내로 명확하게 진술]
    """
)

chain = prompt | model | StrOutputParser()

reports = []
for _, row in failed_cases.iterrows():
    report = chain.invoke({
        "question": row["user_input"],
        "response": row["response"],
        "contexts": row['retrieved_contexts']
    })
    reports.append(report)

failed_cases["report"] = reports

output_csv_path = "generation_failure_report.csv"
failed_cases.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

failed_cases

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy,report
0,도서관 주말 운영 시간과 대출 가능 권수는 어떻게 되나요?,"[중앙도서관 운영 안내: 주말(토, 일)은 오전 10시부터 오후 5시까지 운영하며,...","주말은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",1.0,0.592771,오류 없음
1,시험 기간 평일 도서관 마감 시간은 몇 시인가요?,"[중앙도서관 특별 규정: 시험 기간 평일은 09시부터 22시까지 연장 운영합니다.,...",시험 기간 평일 마감 시간은 22시까지입니다. 또한 주말에는 24시간 특별 개방을 ...,0.5,0.652903,1. 판정 결과: 환각\n2. 상세 원인: 평일 마감 시간 22시는 참고 문서와 일...
